SK_ID_CURR：目前樣本中的貸款編號

NUM_INSTALMENT_VERSION：分期付款計畫版本；版本變動代表付款條件曾調

NUM_INSTALMENT_NUMBER：這筆紀錄所對應的分期期數（第幾期）

DAYS_INSTALMENT：這期原定應付款日，相對於此次貸款申請日（天數）

DAYS_ENTRY_PAYMENT：實際付款日，相對於此次貸款申請日（天數）

AMT_INSTALMENT：此期原本應支付的分期金額

AMT_PAYMENT：客戶實際支付的金額

In [1]:
import pandas as pd
import numpy as np

installments_payments = pd.read_csv("home-credit-default-risk/installments_payments.csv")

In [2]:
missing_rate_percent = (installments_payments.isnull().mean() * 100).sort_values(ascending=False)
print(missing_rate_percent)

DAYS_ENTRY_PAYMENT        0.021352
AMT_PAYMENT               0.021352
SK_ID_PREV                0.000000
SK_ID_CURR                0.000000
NUM_INSTALMENT_VERSION    0.000000
NUM_INSTALMENT_NUMBER     0.000000
DAYS_INSTALMENT           0.000000
AMT_INSTALMENT            0.000000
dtype: float64


In [3]:
# Derive repayment behavior features
installments_payments["DAYS_LATE"] = installments_payments["DAYS_ENTRY_PAYMENT"] - installments_payments["DAYS_INSTALMENT"]
installments_payments["PAYMENT_DIFF"] = installments_payments["AMT_PAYMENT"] - installments_payments["AMT_INSTALMENT"]

installments_payments["PAYMENT_RATIO"] = np.where(
    installments_payments["AMT_INSTALMENT"] > 0,
    installments_payments["AMT_PAYMENT"] / installments_payments["AMT_INSTALMENT"],
    np.nan,
)
installments_payments["LATE_FLAG"] = (installments_payments["DAYS_LATE"] > 0).astype(int)

In [4]:
agg_dict = {
    "NUM_INSTALMENT_NUMBER": ["max"],
    "DAYS_INSTALMENT": ["mean"],
    "DAYS_ENTRY_PAYMENT": ["mean"],
    "AMT_INSTALMENT": ["sum"],
    "AMT_PAYMENT": ["sum"],
    "PAYMENT_DIFF": ["mean"],
    "PAYMENT_RATIO": ["mean"],
    "DAYS_LATE": ["mean"],
    "LATE_FLAG": ["sum"],
}

installments_agg = installments_payments.groupby("SK_ID_CURR").agg(agg_dict)
installments_agg.columns = [f"{col}_{stat}" for col, stat in installments_agg.columns]
installments_agg = installments_agg.reset_index()

installments_agg.head()

,SK_ID_CURR,NUM_INSTALMENT_NUMBER_max,DAYS_INSTALMENT_mean,DAYS_ENTRY_PAYMENT_mean,AMT_INSTALMENT_sum,AMT_PAYMENT_sum,PAYMENT_DIFF_mean,PAYMENT_RATIO_mean,DAYS_LATE_mean,LATE_FLAG_sum
0,100001,4,-2187.714286,-2195.000000,41195.925,41195.925,0.0,1.0,-7.285714,1
1,100002,19,-295.000000,-315.421053,219625.695,219625.695,0.0,1.0,-20.421053,0
2,100003,12,-1378.160000,-1385.320000,1618864.650,1618864.650,0.0,1.0,-7.160000,0
3,100004,3,-754.000000,-761.666667,21288.465,21288.465,0.0,1.0,-7.666667,0
4,100005,9,-586.000000,-609.555556,56161.845,56161.845,0.0,1.0,-23.555556,1


In [5]:
pd.DataFrame.to_csv(installments_agg, "transformed_data/_installments.csv") 